# 02 — Preprocessing Study

Course-aligned (Lecture 1 material). Systematically compare preprocessing pipelines:

1. Baseline: z-score only (already done in prepare_data.py)
2. Histogram Equalization
3. CLAHE (Contrast-Limited Adaptive Histogram Equalization)
4. Gaussian smoothing
5. Median filtering
6. Quantitative comparison: contrast, SNR, edge sharpness

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

BRATS_2D = Path('/kaggle/working/brats2020_2d')

import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import exposure, filters
from scipy.ndimage import median_filter

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load a representative slice with tumor content
with open(BRATS_2D / 'metadata.json') as f:
    meta = json.load(f)
slices = meta['slices']

# Pick a middle slice from the first patient
pid = sorted({s['patient'] for s in slices})[0]
pid_slices = [s for s in slices if s['patient'] == pid]
mid = pid_slices[len(pid_slices) // 2]

img = np.load(BRATS_2D / mid['patient'] / f"slice_{mid['slice']:03d}_image.npy")  # (4,H,W)
msk = np.load(BRATS_2D / mid['patient'] / f"slice_{mid['slice']:03d}_mask.npy")   # (H,W)

# Use FLAIR for visual comparisons (most informative for edema)
flair = img[0]   # z-scored, float32
print(f'FLAIR shape: {flair.shape}  min={flair.min():.2f}  max={flair.max():.2f}')

## Helper: normalise to uint8 for histogram operations

In [ ]:
def to_uint8(arr: np.ndarray) -> np.ndarray:
    """Clip to [0,1] range then scale to [0,255] uint8."""
    a = arr - arr.min()
    if a.max() > 0:
        a = a / a.max()
    return (a * 255).astype(np.uint8)

def to_float(arr: np.ndarray) -> np.ndarray:
    """Scale uint8 back to [0,1] float."""
    return arr.astype(np.float32) / 255.0

flair_u8 = to_uint8(flair)
print(f'uint8: min={flair_u8.min()}  max={flair_u8.max()}')

## 1. Preprocessing Techniques

In [ ]:
# --- Histogram Equalization ---
hist_eq = cv2.equalizeHist(flair_u8)

# --- CLAHE ---
# clipLimit: contrast limit; tileGridSize: local region size
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_img = clahe.apply(flair_u8)

# --- Gaussian smoothing ---
gauss_sigma1 = filters.gaussian(flair, sigma=1.0)
gauss_sigma2 = filters.gaussian(flair, sigma=2.0)

# --- Median filtering ---
median3 = median_filter(flair, size=3)
median5 = median_filter(flair, size=5)

print('All preprocessed variants computed.')

In [ ]:
# Visual comparison grid
variants = [
    ('Baseline (z-score)',    flair,                    True),
    ('Hist EQ',               to_float(hist_eq),        False),
    ('CLAHE (clip=2, 8×8)',   to_float(clahe_img),      False),
    ('Gaussian σ=1',          gauss_sigma1,             True),
    ('Gaussian σ=2',          gauss_sigma2,             True),
    ('Median 3×3',            median3,                  True),
    ('Median 5×5',            median5,                  True),
]

from matplotlib.colors import ListedColormap
MASK_CMAP = ListedColormap([
    (0, 0, 0, 0),
    (0.9, 0.2, 0.2, 0.6),
    (0.2, 0.8, 0.2, 0.6),
    (0.9, 0.9, 0.2, 0.6),
])

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.ravel()

for i, (title, v, is_float) in enumerate(variants):
    ax = axes[i]
    ax.imshow(v, cmap='gray')
    ax.imshow(msk, cmap=MASK_CMAP, vmin=0, vmax=3, interpolation='none')
    ax.set_title(title, fontsize=9)
    ax.axis('off')

axes[-1].axis('off')  # hide unused subplot
fig.suptitle('FLAIR preprocessing comparison — tumor overlay', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Quantitative Metrics

In [ ]:
def michelson_contrast(arr: np.ndarray, brain_mask: np.ndarray) -> float:
    """(max-min)/(max+min) on brain voxels."""
    brain = arr[brain_mask]
    mn, mx = brain.min(), brain.max()
    return float((mx - mn) / (mx + mn + 1e-8))

def snr(arr: np.ndarray, signal_mask: np.ndarray) -> float:
    """Mean/std of signal region."""
    sig = arr[signal_mask]
    return float(sig.mean() / (sig.std() + 1e-8))

def edge_sharpness(arr: np.ndarray) -> float:
    """Mean gradient magnitude via Sobel."""
    u8 = to_uint8(arr)
    gx = cv2.Sobel(u8, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(u8, cv2.CV_64F, 0, 1, ksize=3)
    return float(np.sqrt(gx**2 + gy**2).mean())

brain_mask = flair != 0
tumor_mask = msk > 0

print(f'{"Variant":<25} {"Contrast":>10} {"SNR":>10} {"Edge":>10}')
print('-' * 57)
for title, v, _ in variants:
    c = michelson_contrast(v, brain_mask)
    s = snr(v, tumor_mask) if tumor_mask.any() else float('nan')
    e = edge_sharpness(v)
    print(f'{title:<25} {c:>10.4f} {s:>10.4f} {e:>10.4f}')

In [ ]:
# Histogram comparison: baseline vs CLAHE vs HistEQ
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (title, arr) in zip(axes, [
    ('Baseline (z-score)', flair_u8),
    ('Hist EQ', hist_eq),
    ('CLAHE', clahe_img),
]):
    brain = arr[brain_mask]
    ax.hist(brain, bins=128, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Intensity')
    ax.set_ylabel('Count')

plt.suptitle('Intensity histogram: baseline vs equalization methods')
plt.tight_layout()
plt.show()

## 3. Effect on All 4 Modalities

In [ ]:
# Show CLAHE applied to all modalities
modality_names = ['FLAIR', 'T1', 'T1ce', 'T2']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, (name, ch) in enumerate(zip(modality_names, img)):
    u8 = to_uint8(ch)
    cl = clahe.apply(u8)
    
    axes[0, col].imshow(u8, cmap='gray')
    axes[0, col].imshow(msk, cmap=MASK_CMAP, vmin=0, vmax=3, interpolation='none')
    axes[0, col].set_title(f'{name}\n(baseline)')
    axes[0, col].axis('off')
    
    axes[1, col].imshow(cl, cmap='gray')
    axes[1, col].imshow(msk, cmap=MASK_CMAP, vmin=0, vmax=3, interpolation='none')
    axes[1, col].set_title(f'{name}\n(CLAHE)')
    axes[1, col].axis('off')

plt.suptitle('Baseline vs CLAHE — all 4 modalities', fontsize=12)
plt.tight_layout()
plt.show()

## 4. CLAHE Hyperparameter Study

In [ ]:
# Sweep clip limits and tile sizes
clip_limits = [1.0, 2.0, 4.0]
tile_sizes  = [4, 8, 16]

fig, axes = plt.subplots(len(clip_limits), len(tile_sizes), figsize=(12, 12))

for i, clip in enumerate(clip_limits):
    for j, tile in enumerate(tile_sizes):
        c = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
        out = c.apply(flair_u8)
        axes[i, j].imshow(out, cmap='gray')
        axes[i, j].imshow(msk, cmap=MASK_CMAP, vmin=0, vmax=3, interpolation='none')
        axes[i, j].set_title(f'clip={clip}, tile={tile}×{tile}', fontsize=8)
        axes[i, j].axis('off')

plt.suptitle('CLAHE hyperparameter grid', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

| Method | Pros | Cons | Recommendation |
|---|---|---|---|
| z-score (baseline) | Preserves relative intensities; simple | Range not bounded | Keep as standard |
| Hist EQ | Maximises global contrast | Over-saturates homogeneous regions; destroys tissue contrast in MRI | Avoid |
| CLAHE (clip=2, 8×8) | Local contrast enhancement; limits noise amplification | Slight halo artifacts at boundaries | **Use for EDA/visualisation; optional for training** |
| Gaussian σ=1 | Mild denoising | Blurs edges | May help with noisy slices |
| Median 3×3 | Edge-preserving denoising | Slight smearing | Better than Gaussian for MRI |

**Decision for the pipeline:** Keep z-score normalization as the training-time preprocessing. CLAHE can be applied in the augmentation pipeline as an optional intensity augmentation.